# Fehlendes maxspeed an Hauptstraßen mit Unfällen → MapRoulette

Findet Abschnitte von Hauptstraßen (trunk bis tertiary) ohne `maxspeed` in OpenStreetMap, an
denen Mapillary ein Tempo-Schild erkannt hat, und zählt die Unfälle mit Personenschaden auf jedem
Abschnitt (Unfallatlas, 2020–2025). Ein Abschnitt mit mindestens 4 Unfällen wird eine Aufgabe,
ab 10 Unfällen mit hoher Priorität. Läuft von Hand, wenn eine neue Runde fällig ist.

Aufgebaut wie die `maproulette_tasks.ipynb` der übrigen Kampagnen. Die Schritte liegen in
[`../campaign.py`](../campaign.py); was diese Kampagne ausmacht — Abschnitte, Zuordnung zur
Straße, Unfalldaten, Aufgabentext — in [`../ms_campaign.py`](../ms_campaign.py).

**Ablauf**

| Schritt | Funktion |
| --- | --- |
| Verkehrszeichen von data.vizsim.de spiegeln | `campaign.sync_features` |
| Tempo-Schilder 30–130 einlesen | `campaign.load_features` |
| auf Deutschland beschneiden | `campaign.clip_to_boundary` |
| frische / kurzlebige Schilder aussortieren | `campaign.filter_stable_signs` |
| Hauptstraßen ohne maxspeed zu Abschnitten verbinden | `ms.read_roads`, `ms.hat_maxspeed`, `ms.abschnitte` |
| Schilder an Autobahnen und aus der Tempo-30-Kampagne abziehen | `ms.drop_near_motorways`, `campaign.drop_near_existing_tasks` |
| Schilder und Unfälle der Straße zuordnen, auf der sie liegen | `ms.naechste_hauptstrasse` |
| Unfalldaten holen und prüfen | `ms.sync_unfaelle`, `ms.load_unfaelle` |
| Abschnitte mit Schild und genug Unfällen, Priorität | `ms.abschnitt_aufgaben` |
| bereits erledigte Aufgaben abziehen | `campaign.drop_known_task_names` |
| **neuestes** Mapillary-Bild je Schild holen | `campaign.newest_image_ids`, `ms.zeichen_liste` |
| GeoJSON für MapRoulette schreiben | `campaign.build_maproulette_geojson` mit `ms.aufgabe` |
| in die Challenge hochladen (nur mit `CAMPAIGN_UPLOAD=1`) | `campaign.upload_tasks` |

Die Challenge gibt es noch nicht. Sie holt ihre Aufgaben dann aus
`maproulette_tasks_missing-maxspeed_accidents.geojson` auf GitHub: Notebook laufen lassen, Datei
committen und pushen, in MapRoulette „Rebuild tasks“ (siehe [README](README.md)).

## Einstellungen

In [ ]:
import json
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# campaign.py and ms_campaign.py live one level up. ".." works because Jupyter and
# nbconvert run the kernel in the notebook's directory - the same assumption as
# ../../output/ and ../utils/.
sys.path.insert(0, "..")
import campaign
import ms_campaign as ms

# Must match the date in ../0_prepare_osm_network.ipynb, which writes the roads extract.
set_date = "261017"

ordner_zeichen = Path("../../output")
ordner_unfaelle = Path("../utils/unfallorte")
pfad_strassen = Path(f"../utils/processed_osm_files/processed_roads_germany_{set_date}.parquet")
pfad_grenze = Path("../utils/OSMB-germany.geojson.gz")
pfad_config = Path("../utils/config_mapillary_privat.json")
pfad_t30 = Path("../schools_tempo30_campaign/maproulette_tasks_missing-t30_schools.geojson")

# Only signs last seen on 1 Jan 2025 or later - the dates are YYYY-MM-DD strings,
# and filter_stable_signs keeps what comes after this one ...
zuletzt_gesehen_nach = "2024-12-31"
# ... and seen over at least this many months - filters out temporary signs. At
# main roads with a maxspeed, signs seen over >= 3 months matched it in 73 % of
# cases, signs seen in a single month in 61 %.
mindest_standzeit_monate = 3

# No challenge yet. Once it exists, its id goes here: then the names of its closed
# tasks are not created again, and section 11 can upload.
challenge_id = None
# The Tempo-30 campaign, whose signs are left to it.
t30_challenge_id = 52985

# MapRoulette pulls the tasks from this file on GitHub (remoteGeoJson), so it keeps
# its name.
ziel_geojson = Path("maproulette_tasks_missing-maxspeed_accidents.geojson")

# From the environment, so a run on the server needs no edit to this notebook:
#   CAMPAIGN_SYNC=0    read ../../output/ as it is. On the server it is what the
#                      weekly run produced, and mirroring would overwrite its manifest.
#   CAMPAIGN_UPLOAD=1  upload the tasks to the challenge - refused by MapRoulette
#                      until it has tokens for scripts, see the README.
sync = os.environ.get("CAMPAIGN_SYNC", "1") != "0"
hochladen = os.environ.get("CAMPAIGN_UPLOAD") == "1"

spalten = ["id", "value", "first_seen_at", "last_seen_at", "geometry"]

print(f"Unfälle {ms.UNFALL_JAHRE[0]}-{ms.UNFALL_JAHRE[1]}, Prioritäten {ms.PRIO_AB_UNFAELLE}")
print(f"Zuordnung: Unfall bis {ms.UNFALL_MAX_M:.0f} m, Schild bis {ms.ZEICHEN_MAX_M:.0f} m, "
      f"andere Straße höchstens {ms.NAEHER_ALS_M:.0f} m näher")
print(f"spiegeln: {sync}, hochladen: {hochladen}")

## 1 · Verkehrszeichen holen und einlesen

In [ ]:
if sync:
    # Fetches only what is missing locally or newer on data.vizsim.de.
    campaign.sync_features(ordner_zeichen)

manifest = json.loads((ordner_zeichen / "ml-ts_metadata.json").read_text(encoding="utf-8"))
print("Datenstand:", manifest["processed_date"])
print("im letzten Lauf unvollständig:", manifest.get("last_run_incomplete") or "nichts")

In [ ]:
# Expected number of states from the tile cache; None without one.
erwartet = campaign.count_expected_states("../../prep/tile_cache/DE-*_tiles.json")
print("Tile-Cache kennt", erwartet, "Bundesländer")

signs = campaign.load_features(
    ordner_zeichen,
    prefix=campaign.PREFIX_ZEICHEN,
    values=ms.ZEICHEN,
    columns=spalten,
    expect_files=erwartet,
)
signs = campaign.clip_to_boundary(signs, campaign.load_boundary(pfad_grenze))

## 2 · Zeitliche Filter

In [ ]:
fig, achsen = plt.subplots(1, 2, figsize=(14, 3.5))
signs["last_seen_at"].str[:4].value_counts().sort_index().plot(kind="bar", ax=achsen[0], title="alle Schilder, zuletzt gesehen")

signs = campaign.filter_stable_signs(signs, zuletzt_gesehen_nach, mindest_standzeit_monate)
signs["kmh"] = ms.tempo(signs["value"])

signs["kmh"].value_counts().sort_index().plot(
    kind="bar", ax=achsen[1], title=f"ab 2025 gesehen, ≥ {mindest_standzeit_monate} Monate: Tempo"
)
plt.tight_layout()

## 3 · Hauptstraßen ohne maxspeed und ihre Abschnitte

Ohne `maxspeed` heißt: weder `maxspeed` noch `:forward` noch `:backward`. `maxspeed:type`,
`source:maxspeed` oder `zone:maxspeed` allein zählen nicht — der Wert selbst fehlt.

Ein **Abschnitt** sind verbundene Wege ohne `maxspeed` mit gleichem Namen, ohne Namen mit
gleicher `ref`. Gezählt wird je Abschnitt und nicht je OSM-Weg: In Städten ist eine Straße oft in
Stücke von 30 m zerteilt, und das Stück mit dem Schild trägt selten die Unfälle.

In [ ]:
strassen = ms.read_roads(pfad_strassen)
ohne_maxspeed = strassen["highway"].isin(ms.HAUPTSTRASSEN) & ~ms.hat_maxspeed(strassen)

wege = strassen[ohne_maxspeed].copy()
wege["abschnitt"] = ms.abschnitte(wege)
# Section of each row of `strassen`; -1 for roads that are no candidate.
abschnitt_je_strasse = np.full(len(strassen), -1, dtype=np.int64)
abschnitt_je_strasse[wege.index.to_numpy()] = wege["abschnitt"].to_numpy()

haupt_km = strassen.geometry[strassen["highway"].isin(ms.HAUPTSTRASSEN)].length.sum() / 1000
ohne_km = wege.geometry.length.sum() / 1000
print(f"Hauptstraßen: {ms.zahl(haupt_km)} km, ohne maxspeed: {ms.zahl(ohne_km)} km "
      f"({100 * ohne_km / haupt_km:.1f} %)")
print(f"Abschnitte ohne maxspeed: {ms.zahl(wege['abschnitt'].nunique())}")

## 4 · Schilder der Straße zuordnen

Ein Schild gehört zur nächsten Hauptstraße, höchstens 20 m entfernt — außer eine andere Straße
liegt mehr als 3 m näher: Dann steht es an dieser (`ms.naechste_hauptstrasse`). Schilder bis 30 m
an einer Autobahn fallen weg, meist Autobahnschilder von einer Brücke aus gesehen. Ebenso Schilder
bis 30 m an einer Aufgabe der Tempo-30-Kampagne, gleich in welchem Status: Die gehören dort hin.

In [ ]:
signs = ms.drop_near_motorways(signs.to_crs(strassen.crs), strassen)

position = ms.naechste_hauptstrasse(signs, strassen, ms.ZEICHEN_MAX_M)
signs["abschnitt"] = np.where(position >= 0, abschnitt_je_strasse[np.maximum(position, 0)], -1)
print(f"an einer Hauptstraße: {ms.zahl((position >= 0).sum())}, davon ohne maxspeed: {ms.zahl((signs['abschnitt'] >= 0).sum())}")
signs = signs[signs["abschnitt"] >= 0].reset_index(drop=True)

api_key = campaign.load_token(pfad_config, "API_KEY_MAPROULETTE")
t30_aufgaben = pd.concat([
    campaign.load_challenge_tasks(t30_challenge_id, api_key, statuses_to_keep=campaign.TASK_STATUS)[["geometry"]],
    gpd.read_file(pfad_t30)[["geometry"]],
], ignore_index=True)
print(f"Aufgaben der Tempo-30-Kampagne (Challenge und aktuelle Datei): {ms.zahl(len(t30_aufgaben))}")
signs = campaign.drop_near_existing_tasks(
    signs, gpd.GeoDataFrame(t30_aufgaben, crs=4326), radius_m=ms.T30_ABSTAND_M
)
print(f"Schilder an Hauptstraßen ohne maxspeed: {ms.zahl(len(signs))}")

## 5 · Unfälle

Die Datei von data.vizsim.de mit dem Stand im Namen (`latest.json` → `datei`), geprüft gegen ihre
SHA-256. Nur 2020–2025: Erst ab 2020 sind alle Länder im Unfallatlas, `ms.load_unfaelle` bricht ab,
wenn in einem dieser Jahre ein Land fehlt. Zugeordnet wird wie bei den Schildern, bis 15 m.

In [ ]:
pfad_unfaelle, unfall_stand = ms.sync_unfaelle(ordner_unfaelle)
print("Stand der Unfalldaten:", unfall_stand["version"])
unfaelle = ms.load_unfaelle(pfad_unfaelle)

position = ms.naechste_hauptstrasse(unfaelle, strassen, ms.UNFALL_MAX_M)
unfaelle["abschnitt"] = np.where(position >= 0, abschnitt_je_strasse[np.maximum(position, 0)], -1)
an_haupt = position >= 0
ohne = unfaelle["abschnitt"] >= 0
print(f"an Hauptstraßen: {ms.zahl(an_haupt.sum())} ({100 * an_haupt.mean():.1f} %), "
      f"davon auf Abschnitten ohne maxspeed: {ms.zahl(ohne.sum())} ({100 * ohne.sum() / an_haupt.sum():.1f} %)")
unfaelle = unfaelle[ohne]

## 6 · Aufgaben

Ein Abschnitt mit mindestens einem Schild und mindestens 4 Unfällen wird eine Aufgabe:
Priorität **hoch** ab 10 Unfällen, sonst **mittel** (`ms.PRIO_AB_UNFAELLE`). Name der Aufgabe ist
die kleinste OSM-Weg-ID des Abschnitts.

In [ ]:
aufgaben = ms.abschnitt_aufgaben(wege, signs, unfaelle)
del strassen, wege

print(aufgaben["prio"].map(ms.PRIO_TEXT).value_counts().to_string())
print("Länge (km):", aufgaben["km"].quantile([0.1, 0.5, 0.9]).round(2).to_dict())
aufgaben[["id", "strasse", "highway", "km", "unfaelle", "unfaelle_schwer", "tempo", "prio"]].head(10)

## 7 · Bereits erledigte Aufgaben abziehen

Erst, wenn es die Challenge gibt: Ein Abschnitt, dessen Name eine erledigte Aufgabe ist (Fixed,
Not an Issue, …), würde diese nur aktualisieren und bliebe geschlossen.

In [ ]:
if challenge_id is not None:
    aufgaben = campaign.drop_known_task_names(aufgaben, campaign.challenge_task_names(challenge_id, api_key))
else:
    print("noch keine Challenge - nichts abzuziehen")
print(f"Aufgaben: {len(aufgaben)}")

## 8 · Neuestes Mapillary-Bild je Schild

Eine Aufgabe zeigt bis zu `ms.MAX_ZEICHEN_JE_AUFGABE` (5) Schilder, die zuletzt gesehenen zuerst.

In [ ]:
gezeigt = [i for ids in aufgaben["zeichen_ids"] for i in ids[: ms.MAX_ZEICHEN_JE_AUFGABE]]
bilder = campaign.newest_image_ids(gezeigt, campaign.load_token(pfad_config))
aufgaben["zeichen_liste"] = ms.zeichen_liste(aufgaben, signs, bilder)

### Plausibilitätsprüfung

`last_seen_at` aus dem Parquet ist die letzte Aufnahme, auf der das Schild erkannt wurde. Stimmt
die Bildauswahl, ist das verlinkte Bild ungefähr von diesem Tag.

In [ ]:
gezeigte = pd.DataFrame([z for liste in aufgaben["zeichen_liste"] for z in liste])
abstand_tage = (
    pd.to_datetime(gezeigte["image_captured_at"], utc=True).dt.tz_localize(None) - pd.to_datetime(gezeigte["last_seen_at"])
).dt.days.abs()
print(abstand_tage.describe().to_string())
print(f"\nBild am selben Tag wie last_seen_at: {(abstand_tage <= 1).sum()} von {abstand_tage.notna().sum()}")

## 9 · GeoJSON für MapRoulette schreiben

Eine Aufgabe ist der Abschnitt als Linie. Die Datei behält ihren Namen, weil MapRoulette sie von
GitHub holt. Ihr bisheriger Inhalt wird **vor** dem Überschreiben eingelesen.

In [ ]:
vorheriger_stand = campaign.read_geojson(ziel_geojson)

sammlung = campaign.build_maproulette_geojson(aufgaben, ms.aufgabe)
campaign.write_geojson(sammlung, ziel_geojson)

if sammlung["features"]:
    print(sammlung["features"][0]["properties"]["instruction"])

## 10 · Vergleich mit dem vorherigen Stand

In [ ]:
if vorheriger_stand:
    campaign.print_comparison(campaign.compare_task_sets(vorheriger_stand, sammlung))
else:
    print("kein vorheriger Stand vorhanden - Vergleich übersprungen")

## 11 · In die Challenge hochladen

Direkt über die MapRoulette-API (`addFileTasks`), ohne Commit und ohne „Rebuild tasks“. Läuft nur
mit `CAMPAIGN_UPLOAD=1` und einer `challenge_id`, und **noch gar nicht**: Seit maproulette-backend
v4.11.0 (Oktober 2026) lehnt MapRoulette API-Keys ab. Bis dahin: Datei committen, pushen,
„Rebuild tasks“.

In [ ]:
if hochladen and challenge_id is not None:
    nachher = campaign.upload_tasks(
        challenge_id, sammlung, api_key, data_origin_date=manifest["processed_date"][:10]
    )
    print(pd.Series(nachher).value_counts().to_string())
else:
    print(f"nicht hochgeladen ({len(sammlung['features'])} Aufgaben) - dafür challenge_id und CAMPAIGN_UPLOAD=1 setzen")

---

## Challenge-Beschreibung (für MapRoulette)

## 🚦 Tempolimits an Hauptstraßen mit vielen Unfällen ergänzen (Deutschland)

Diese Challenge zeigt **Abschnitte von Hauptstraßen**, für die in OpenStreetMap noch kein
Tempolimit (`maxspeed`) eingetragen ist, an denen **Mapillary ein Tempo-Schild** erkannt hat und
auf denen es **viele Unfälle mit Personenschaden** gab. Je mehr Unfälle, desto höher die Priorität.

### 📌 Kriterien für jede Aufgabe

- Der Abschnitt ist eine `trunk`-, `primary`-, `secondary`- oder `tertiary`-Straße (inkl. `_link`)
  **ohne** `maxspeed`, `maxspeed:forward` und `maxspeed:backward`.
- Auf dem Abschnitt hat Mapillary ein **Tempo-Schild** (30–130 km/h) erkannt, über **mindestens
  3 Monate** wiederholt und **zuletzt 2025 oder später**.
- Auf dem Abschnitt gab es 2020–2025 **mindestens 4 Unfälle mit Personenschaden** laut
  Unfallatlas; **ab 10 Unfällen** hat die Aufgabe hohe Priorität.

---

### 🔍 Was du tun solltest

1. Öffne die Schilder über die Links in der Aufgabe in **Mapillary** und prüfe sie samt
   **Zusatzzeichen** (Uhrzeit, Lkw, Nässe …).
2. Trage `maxspeed` für den **ganzen Abschnitt** ein, nicht nur am Schild. Wo kein Schild steht,
   gilt innerorts meist 50 (`maxspeed:type=DE:urban`), außerorts 100 (`maxspeed:type=DE:rural`).
3. Zeitabhängige Limits gehören in `maxspeed:conditional`, z. B. `30 @ (22:00-06:00)`;
   richtungsabhängige in `maxspeed:forward` / `maxspeed:backward`.
4. Baustellen oder falsch erkannte Schilder bitte als **„Not an Issue“** markieren.

---

### 📚 Nützliche Links

- [**OSM-Wiki: maxspeed**](https://wiki.openstreetmap.org/wiki/DE:Key:maxspeed)
- [**Traffic Sign Tool**](https://trafficsigns.osm-verkehrswende.org/DE?signs=DE:274-70)

---

Unfalldaten: Unfallatlas, © Statistische Ämter des Bundes und der Länder,
Datenlizenz Deutschland – Namensnennung – Version 2.0 (www.govdata.de/dl-de/by-2-0),
https://unfallatlas.statistikportal.de/. Daten aufbereitet von vizsim (zusammengeführt und
vereinheitlicht), https://data.vizsim.de/unfallorte/; für diese Challenge je Straßenabschnitt gezählt.

---

**Prioritätsregeln** (Challenge → Priorität): High, wenn die Property `priority` gleich `0` ist;
Medium, wenn sie `1` ist.

Task-Template für MapRoulette (Feld *Instruction*):

```
{{instruction}}
```